# Prepara??o dos dados de ransomware

Este notebook valida, converte e prepara o dataset antes da modela??o.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "ransom.csv"
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")
print(f"Existe: {DATA_PATH.is_file()}")

## 1. Inspe??o inicial

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dimens?o inicial: {df.shape}")
display(df.head())
display(df.dtypes.value_counts())

## 2. Qualidade dos dados

In [ ]:
quality = pd.DataFrame({
    "valores_em_falta": df.isna().sum(),
    "percentagem_em_falta": df.isna().mean() * 100,
}).sort_values("percentagem_em_falta", ascending=False)
display(quality.head(30))
print(f"Linhas duplicadas exatas: {df.duplicated().sum()}")
print(f"Linhas duplicadas por MD5: {df.duplicated(subset=['md5']).sum() if 'md5' in df.columns else 'coluna md5 n?o dispon?vel'}")

## 3. Prepara??o pelo m?dulo do projeto

In [ ]:
from src.data_preprocessing import load_and_deduplicate, convert_numeric_like_columns, TARGET, feature_columns

prepared = load_and_deduplicate(DATA_PATH)
prepared = convert_numeric_like_columns(prepared)
print(f"Dimens?o ap?s prepara??o: {prepared.shape}")
print(f"Alvo: {TARGET}")
print(f"N?mero de features: {len(feature_columns(prepared))}")
display(prepared.head())

## 4. Distribui??o da vari?vel alvo

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 4))
sns.countplot(data=prepared, x=TARGET)
plt.title("Distribui??o da classe")
plt.tight_layout()
plt.show()

## 5. Exportar prepara??o resumida

In [ ]:
output_dir = PROJECT_ROOT / "results" / "notebooks" / "data-preparation"
output_dir.mkdir(parents=True, exist_ok=True)
prepared.to_csv(output_dir / "prepared_ransom.csv", index=False)
print(output_dir / "prepared_ransom.csv")